# 00 · Hello, GPU

**Pre-work for the PyCon Italy 2026 workshop** _Write Your First High-Performance GPU Kernel in Python!_

Run every cell **before the day of the workshop**. If you see the green ✓ at the bottom, you are ready. If anything fails, reply to the pre-work email and we will sort it out.

**Expected runtime:** Google Colab, T4 GPU. Set the runtime via _Runtime → Change runtime type → T4 GPU_.

## 1. Environment bootstrap (installs anything missing)

In [ ]:
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    name = import_name or pip_name
    try:
        importlib.import_module(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

ensure('triton')
_cupy_ok = True
try:
    ensure('cupy-cuda12x', 'cupy')
    import cupy
except Exception as exc:
    _cupy_ok = False
    print(f'cupy unavailable (only used in notebook 01 demo): {exc}')

import torch, triton
print(f'torch   {torch.__version__}')
print(f'triton  {triton.__version__}')
if _cupy_ok:
    print(f'cupy    {cupy.__version__}')
print(f'cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'gpu     {torch.cuda.get_device_name(0)}')
    print(f'cc      {torch.cuda.get_device_capability(0)}')
    print(f'mem     {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB')
assert torch.cuda.is_available(), (
    'No CUDA GPU detected. On Colab: Runtime > Change runtime type > T4 GPU.'
)


## 2. Tiny Triton kernel (vector add)

In [ ]:
import torch, triton
import triton.language as tl

@triton.jit
def _add_kernel(a_ptr, b_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(axis=0)
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < n
    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)
    tl.store(out_ptr + offsets, a + b, mask=mask)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
a = torch.arange(1024, device=device, dtype=torch.float32)
b = torch.ones_like(a)
out = torch.empty_like(a)
grid = (triton.cdiv(a.numel(), 256),)
_add_kernel[grid](a, b, out, a.numel(), BLOCK=256)
torch.testing.assert_close(out, a + b)
print('triton vector add: ok')


## 3. Tiny CuPy RawKernel (vector add in CUDA C)

In [ ]:
if not _cupy_ok:
    print('cupy unavailable on this runtime; skipping the RawKernel demo. nb01 is a speaker demo only.')
else:
    import cupy as cp

    src = r'''
extern "C" __global__
void vec_add(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}
'''

    kernel = cp.RawKernel(src, 'vec_add')
    n = 1024
    a = cp.arange(n, dtype=cp.float32)
    b = cp.ones(n, dtype=cp.float32)
    c = cp.empty_like(a)
    threads = 256
    blocks = (n + threads - 1) // threads
    kernel((blocks,), (threads,), (a, b, c, n))
    cp.testing.assert_allclose(c, a + b)
    print('cupy raw vector add: ok')


## 4. You are ready

In [ ]:
print('\n✓ All checks passed. See you in Bologna.')
